<a href="https://colab.research.google.com/github/Mwahyudi19/Laptop-Specs-Explorer-Pricing-Dashboard/blob/main/priceoye_laptops_clean_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import re

df = pd.read_csv('priceoye_laptops_version_2.csv')

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# 1. MISSING VALUES & HARGA

df['Rating'] = df['Rating'].fillna(0)
df['Reviews'] = df['Reviews'].fillna(0)
df['Actual Price'] = df['Actual Price'].fillna(df['Discounted Price'])
df['Saving'] = df['Saving'].fillna('0% OFF')
df['Saving'] = df['Saving'].str.replace('% OFF', '', regex=False).str.strip()
df['Saving'] = pd.to_numeric(df['Saving'], errors='coerce').fillna(0).astype(int)

# 2. EKSTRAKSI FITUR DARI 'Name'

def extract_ram(name):
    name = str(name)
    # Menangkap kasus penulisan aneh seperti "08GB RAM"
    match_ram = re.search(r'(\d+)\s*GB\s*RAM', name, re.IGNORECASE)
    if match_ram: return str(int(match_ram.group(1))) + 'GB'

    # Menangkap kasus format "8-512GB"
    match_edge = re.search(r'\b(\d{1,2})-(\d{3,}GB|1TB)\b', name, re.IGNORECASE)
    if match_edge: return match_edge.group(1) + 'GB'

    # Menangkap format standar (8GB-256GB)
    match = re.search(r'\b(\d+)\s*GB(?!\s*SSD|\s*HDD)', name, re.IGNORECASE)
    if match: return match.group(1).upper() + 'GB'

    # Jika tidak ada satupun yang cocok, tandai sebagai "Unknown"
    return 'Unknown'

df['RAM'] = df['Name'].apply(extract_ram)

def extract_storage(name):
    name = str(name)
    match = re.search(r'\b\d+GB\s*[-|]?\s*(\d+(?:GB|TB))\b', name, re.IGNORECASE)
    if match: return match.group(1).upper() + " SSD"

    match_explicit = re.search(r'\b(\d+(?:GB|TB))\s*(?:SSD|HDD)\b', name, re.IGNORECASE)
    if match_explicit: return match_explicit.group(1).upper() + " SSD"

    match_edge = re.search(r'\b\d{1,2}-(\d{3,}GB|1TB)\b', name, re.IGNORECASE)
    if match_edge: return match_edge.group(1).upper() + " SSD"

    return 'Unknown'

def clean_existing_ssd(ssd_val):
    if pd.isna(ssd_val): return None
    match = re.search(r'(\d+(?:GB|TB))\s*(?:SSD|HDD)', str(ssd_val), re.IGNORECASE)
    if match: return match.group(1).upper() + " SSD"
    return None

df['SSD_Clean'] = df['SSD'].apply(clean_existing_ssd)
df['Extracted_SSD'] = df['Name'].apply(extract_storage)
df['SSD'] = df['SSD_Clean'].fillna(df['Extracted_SSD']).replace('Unknown SSD', None).fillna('Unknown SSD')
df = df.drop(columns=['SSD_Clean', 'Extracted_SSD'])

def extract_core(name):
    name = str(name)
    patterns = [
        r'(?i)(Core\s*i\d)', r'(?i)(Ultra\s*\d)', r'(?i)(Ryzen\s*\d)',
        r'(?i)(R\d-\w+)', r'(?i)\b(M[1-4])\b', r'(?i)(Ci\d)',
    ]
    for pattern in patterns:
        match = re.search(pattern, name)
        if match:
            core_val = match.group(1).upper()
            if core_val.startswith('CI') or core_val.startswith('CORE I'):
                return 'CORE I' + core_val[-1]
            elif core_val.startswith('ULTRA'): return 'ULTRA ' + core_val[-1]
            elif core_val.startswith('RYZEN'): return 'RYZEN ' + core_val[-1]
            elif core_val.startswith('R'): return 'RYZEN ' + core_val[1]
            return core_val
    return 'Unknown'

df['Core'] = df['Core'].astype(str).str.upper().replace('NAN', None)
df['Extracted_Core'] = df['Name'].apply(extract_core)
df['Core'] = df['Core'].fillna(df['Extracted_Core']).replace('UNKNOWN', None).fillna('Unknown')
df = df.drop(columns=['Extracted_Core'])

# 3. PEMBERSIHAN TOTAL NAMA PRODUK (NAME & MODEL)

def clean_product_name(text):
    text = str(text)
    # Hapus semua text yang ada di dalam tanda kurung
    text = re.sub(r'\(.*?\)', '', text)

    # Hapus spesifikasi dari nama
    patterns_to_remove = [
        r'(?i)Intel Core Ultra \d+\s*\w*', r'(?i)Core i\d+', r'(?i)\d{1,2}th Gen',
        r'(?i)Ryzen\s*\d+\s*[-A-Z0-9]*', r'(?i)R\d-\w+', r'(?i)M\d+\s*Chip', r'(?i)\bM[1-4]\b',
        r'(?i)Ci\d+-\w+', r'(?i)Ci\d+', r'(?i)\d+\.\d+\s*Inches',
        r'(?i)\d+\s*Inches', r'(?i)RTX[- ]?\d+[- ]?\d*GB', r'(?i)FHD|WQXGA|DOS',
        r'(?i)Notebook PC', r'(?i)Processor', r'(?i)Intel®|Core™|Intel', r'[\|\-\(\)®™]', r'(?i)\b\d-\d{3,}GB\b'
    ]
    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text)

    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    return cleaned_text.upper()

df['Name'] = df['Name'].apply(clean_product_name)
df['Model'] = df['Model'].apply(clean_product_name)

# 4. SIMPAN HASIL KE CSV BARU

# Mengatur ulang urutan kolom
df = df[['Brand', 'Name', 'Model', 'Core', 'RAM', 'SSD', 'Discounted Price', 'Actual Price', 'Saving', 'Rating', 'Reviews']]

output_filename = 'priceoye_laptops_ready_to_analyze.csv'
df.to_csv(output_filename, index=False)

# Summary Hasil
print("✅ Pembersihan Data Selesai Sepenuhnya!")
print(f"Data tersimpan sebagai: {output_filename}")
print(f"-> Sisa Data RAM yang 'Unknown': {(df['RAM'] == 'Unknown').sum()} baris")
print(f"-> Sisa Data SSD yang 'Unknown': {(df['SSD'] == 'Unknown SSD').sum()} baris")
print("\n--- HASIL AKHIR ---")
display(df.head(10))

✅ Pembersihan Data Selesai Sepenuhnya!
Data tersimpan sebagai: priceoye_laptops_ready_to_analyze.csv
-> Sisa Data RAM yang 'Unknown': 56 baris
-> Sisa Data SSD yang 'Unknown SSD': 0 baris

--- HASIL AKHIR ---


,Brand,Name,Model,Core,RAM,SSD,Discounted Price,Actual Price,Saving,Rating,Reviews
0,Apple,APPLE MACBOOK AIR 13 MGN63,MACBOOK AIR 13 MGN63,M1,8GB,256GB SSD,203499.0,258000.0,21,5.0,2.0
1,Apple,APPLE MACBOOK AIR 13 MW123,MACBOOK AIR 13 MW123,M4,Unknown,Unknown,281999.0,350000.0,19,4.3,5.0
2,ASUS,ASUS ZENBOOK 14 UX3405CA,ZENBOOK 14 UX3405CA,ULTRA 7,16GB,512GB SSD,285999.0,330000.0,13,0.0,0.0
3,Lenovo,LENOVO THINKPAD E16 GEN 2,THINKPAD E16 GEN 2,ULTRA 7,16GB,512GB SSD,287999.0,340000.0,15,0.0,0.0
4,Lenovo,LENOVO IDEAPAD SLIM 3,IDEAPAD SLIM 3,RYZEN 7,8GB,512GB SSD,146999.0,175000.0,16,0.0,0.0
5,HP,HP 250 G10 N305,250 G10 N305,CORE I3,8GB,512GB SSD,115999.0,144999.0,20,0.0,0.0
6,Dell,DELL LATITUDE 3540 15.6,LATITUDE 3540 15.6,CORE I5,8GB,256GB SSD,169999.0,249999.0,32,5.0,1.0
7,Dell,DELL LATITUDE 15 5540,LATITUDE 15 5540,CORE I7,8GB,512GB SSD,343999.0,415000.0,17,0.0,0.0
8,Dell,DELL LATITUDE 15 3540,LATITUDE 15 3540,CORE I7,8GB,512GB SSD,251999.0,320000.0,21,0.0,0.0
9,Lenovo,LENOVO IDEAPAD SLIM 3,IDEAPAD SLIM 3,CORE I5,Unknown,Unknown,134999.0,174999.0,23,5.0,2.0
